# How to use external MatGL and MACE models to select structures from a `pool_set`

## What you need to change

For the MatGL demo, set `pool_set` and choose a `matgl:` model spec such as
`matgl:TensorNet-PES-MatPES-PBE-2025.2-m`.

For the full MACE selection at the end, the notebook uses an official MACE foundation-model URL by default, so it can run even when no local MACE checkpoint has been downloaded. You can replace `mace_model_url` with another official `.model` URL if needed.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / 'curator').is_dir() else cwd.parent
notebook_dir = repo_root / 'notebooks'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Change only this path.
pool_set = repo_root / 'example' / 'LiFePO4.traj'
assert pool_set.exists(), f'pool_set does not exist: {pool_set}'

run_dir = notebook_dir / 'runs' / 'matgl_tensornet_2025_2_m_select_demo'
run_dir.mkdir(parents=True, exist_ok=True)

pool_set, run_dir


## Imports

`load_models(...)` is the key entry point.
When CURATOR sees a string that starts with `matgl:`, it loads the MatGL model and wraps it in a CURATOR-compatible adapter.


In [ ]:
import inspect
import json
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch

from ase.io import read, write
from pytorch_lightning import seed_everything
from umap import UMAP

from curator.layer._feature import FeatureExtractor
from curator.layer.utils import find_layer_by_name_recursive
from curator.select import GeneralActiveLearning
from curator.utils import load_models

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
seed_everything(123, workers=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


## 1. Load the MatGL model

This is the exact idea used by `curator-select`:

- pass a `model_path` string,
- let `load_models(...)` decide how to load it,
- get back a CURATOR model object.

For a MatGL model, the returned object is a `MatGLAdapter`, not the raw MatGL model.


In [ ]:
model = load_models(
    'matgl:TensorNet-PES-MatPES-PBE-2025.2-m',
    device=device,
    load_compiled=False,
)[0]

{
    'loaded_object': model.__class__.__name__,
    'wrapped_core_model': model.core_model.__class__.__name__,
    'backend': model.backend,
    'cutoff': model.representation.cutoff,
    'device': device,
    'load_models_source': inspect.getfile(load_models),
    'adapter_source': inspect.getfile(model.__class__),
}


## 2. Check the target layer used by selection

For this TensorNet example, selection uses `target_layer='final_layer'`.

That matters because `FeatureExtractor` will attach hooks under this module and collect the features and gradients used by the selection kernel.


In [ ]:
target_layer = 'final_layer'
target_module = find_layer_by_name_recursive(model, target_layer)
linear_types = FeatureExtractor._resolve_linear_types()
linear_modules = [
    (name, child.__class__.__name__)
    for name, child in target_module.named_modules()
    if isinstance(child, linear_types)
]

print('target layer name:', target_layer)
print('resolved module class:', target_module.__class__.__name__)
print('number of linear-like submodules:', len(linear_modules))
linear_modules[:8]


## 3. See how one structure from `pool_set` becomes a MatGL graph

`MatGLAdapter` converts ASE atoms into the graph objects expected by MatGL.
This is the bridge between CURATOR's dataset pipeline and the MatGL model.


In [ ]:
atoms = read(pool_set, index=0)
graph, lat, state_attr = model.graph_converter.get_graph(atoms)

num_nodes = graph.num_nodes if not callable(getattr(graph, 'num_nodes', None)) else graph.num_nodes()
try:
    num_edges = graph.num_edges if not callable(getattr(graph, 'num_edges', None)) else graph.num_edges()
except Exception:
    num_edges = int(graph.edge_index.shape[1])

{
    'formula': atoms.get_chemical_formula(),
    'num_atoms': len(atoms),
    'pbc': tuple(bool(x) for x in atoms.pbc),
    'graph_class': graph.__class__.__name__,
    'graph_nodes': num_nodes,
    'graph_edges': num_edges,
    'lat_shape': tuple(lat.shape),
    'state_attr_shape': tuple(state_attr.shape),
}


## 4. Build the active-learning object

This object is what actually drives structure selection.

Important arguments here:

- `models=[model]`: use the MatGL TensorNet adapter
- `feature_specs=[{'preset': 'llg-sketch'}]`: use sketched last-layer-gradient features
- `selection_feature='llg-sketch'`: select with that exported feature block
- `selection='lcmd_greedy'`: use LCMD greedy selection first
- `target_layer='final_layer'`: extract features from TensorNet's final readout layer
- `save_features=...`: store the features so they can be reused by DIRECT and visualized later


In [ ]:
features_h5 = run_dir / 'features_llg.h5'
feature_specs = [
    {
        'preset': 'llg-sketch',
        'num_features': 256,
        'seed': 123,
    }
]
selection_feature = 'llg-sketch'
direct_selection_kwargs = {
    'n_clusters': 100,
    'k': 1,
    'threshold': 0.5,
    'weighting_pcs': True,
    'selection_criteria': 'center',
}

al = GeneralActiveLearning(
    models=[model],
    selection='lcmd_greedy',
    feature_specs=feature_specs,
    selection_feature=selection_feature,
    target_layer='final_layer',
    batch_size=4,
    device=device,
    dataset_cutoff=model.representation.cutoff,
    transforms=[],
    save_features=str(features_h5),
    target_domain=None,
)

{
    'selection_engine': al.__class__.__name__,
    'selection_engine_source': inspect.getfile(GeneralActiveLearning),
    'feature_specs': feature_specs,
    'selection_feature': al.selection_feature,
    'selection_method': al.selection,
    'direct_selection_kwargs': direct_selection_kwargs,
    'target_layer': al.target_layer,
    'batch_size': al.batch_size,
    'dataset_cutoff': al.dataset_cutoff,
    'features_h5': str(features_h5),
}


## 5. Select structures from `pool_set`

This is the core call.

If your `pool_set` is large, this may take some time. For a first test, you can lower `select_batch_size`.
The selected indices are written to `selected_lcmd_greedy.json` and `selected_direct_birch.json`.


In [ ]:
select_batch_size = 100

selected = al.select(
    pool_set=str(pool_set),
    train_set=None,
    select_batch_size=select_batch_size,
    save_json=str(run_dir / 'selected_lcmd_greedy.json'),
    save_images=None,
    save_selected_features=None,
    normalize_features=True,
    compute_features_only=False,
)

direct_al = GeneralActiveLearning(
    models=[model],
    selection='direct_birch',
    selection_kwargs=direct_selection_kwargs,
    feature_specs=feature_specs,
    selection_feature=selection_feature,
    target_layer='final_layer',
    batch_size=4,
    device=device,
    dataset_cutoff=model.representation.cutoff,
    transforms=[],
    save_features=str(features_h5),
    target_domain=None,
)
direct_selected = direct_al.select(
    pool_set=str(pool_set),
    train_set=None,
    select_batch_size=select_batch_size,
    save_json=str(run_dir / 'selected_direct_birch.json'),
    save_images=None,
    save_selected_features=None,
    normalize_features=True,
    compute_features_only=False,
)

print('LCMD selected structures:', len(selected))
print('LCMD first ten selected indices:', selected[:10])
print('DIRECT selected structures:', len(direct_selected))
print('DIRECT first ten selected indices:', direct_selected[:10])


In [ ]:
{
    'lcmd_greedy': json.loads((run_dir / 'selected_lcmd_greedy.json').read_text()),
    'direct_birch': json.loads((run_dir / 'selected_direct_birch.json').read_text()),
}


## 6. Visualize the selection in UMAP space

The plotting logic below is adapted from:
`/home/xinyang/curator/runs/select_matgl_compare/plot_selection_umap_with_aef.py`

The idea is:

1. load the saved `llg-sketch` features from `features_llg.h5`,
2. reduce them to 2D with UMAP,
3. overlay the CURATOR `lcmd_greedy` and `direct_birch` selections,
4. build a simple baseline that selects the structures with the largest AE-F,
5. compare all selections on the same figure,
6. if reference forces are available in `pool_set`, color the background by AE-F
   (mean absolute force error).

AE-F needs reference forces inside the structures in `pool_set`.
If they are not available, the notebook will still make a UMAP plot, but the
`max AE-F` comparison will not be available.


In [ ]:
def read_selected(path: Path) -> np.ndarray:
    with path.open('r', encoding='utf-8') as handle:
        return np.asarray(json.load(handle)['selected'], dtype=int)


def has_reference_forces(path: Path) -> bool:
    sample = read(path, index=0)
    try:
        sample.get_forces(apply_constraint=False)
        return True
    except Exception:
        return False


def compute_aef_from_adapter(model, pool_set: Path, batch_size: int = 8) -> np.ndarray:
    atoms_list = read(str(pool_set), ':')
    aef = np.zeros((len(atoms_list),), dtype=np.float64)

    for start in range(0, len(atoms_list), batch_size):
        batch = atoms_list[start : start + batch_size]
        n_atoms = [len(atoms) for atoms in batch]
        g, lat, state = model._batch_graphs(batch)
        g = g.to(device)
        lat = lat.to(device)
        state = state.to(device)

        out = model.potential(g=g, lat=lat, state_attr=state)
        if not isinstance(out, tuple) or len(out) < 2:
            raise RuntimeError('MatGL potential output does not include force predictions.')
        force_pred = out[1].detach().cpu().numpy()

        offset = 0
        for i, atoms in enumerate(batch):
            n = n_atoms[i]
            fp = force_pred[offset : offset + n]
            ft = atoms.get_forces(apply_constraint=False)
            aef[start + i] = float(np.mean(np.abs(fp - ft)))
            offset += n

    return aef


def compute_aef_from_curator_model(model, pool_set: Path, batch_size: int = 4) -> np.ndarray:
    from curator.data import AseDataset, properties
    from curator.data.utils import iter_batches

    dataset = AseDataset(str(pool_set), cutoff=model.representation.cutoff)
    model_device = next(model.parameters()).device
    model_dtype = next(model.parameters()).dtype
    aef = np.zeros((len(dataset),), dtype=np.float64)
    image_offset = 0
    was_training = model.training
    model.eval()

    for batch in iter_batches(
        dataset=dataset,
        batch_size=batch_size,
        device=model_device,
        dtype=model_dtype,
        desc=f'AE-F model={model.__class__.__name__} size={len(dataset)} bs={batch_size}',
    ):
        target_forces = batch[properties.forces].detach().cpu().numpy()
        predicted = model(batch)[properties.forces].detach().cpu().numpy()
        counts = batch[properties.n_atoms].detach().cpu().view(-1).tolist()

        atom_offset = 0
        for n_atoms in counts:
            n_atoms = int(n_atoms)
            fp = predicted[atom_offset : atom_offset + n_atoms]
            ft = target_forces[atom_offset : atom_offset + n_atoms]
            aef[image_offset] = float(np.mean(np.abs(fp - ft)))
            image_offset += 1
            atom_offset += n_atoms

    if was_training:
        model.train()
    return aef


def summarize_selection_overlap(lcmd_idx: np.ndarray, direct_idx: np.ndarray, aef: np.ndarray | None = None) -> dict:
    lcmd_set = set(lcmd_idx.tolist())
    direct_set = set(direct_idx.tolist())
    overlap = lcmd_set & direct_set
    union = lcmd_set | direct_set
    summary = {
        'LCMD count': len(lcmd_idx),
        'DIRECT count': len(direct_idx),
        'LCMD-DIRECT overlap': len(overlap),
        'LCMD-DIRECT Jaccard': len(overlap) / len(union) if union else 0.0,
    }
    if aef is not None:
        max_aef_set = set(np.argsort(aef)[-len(lcmd_idx):].tolist())
        summary.update(
            {
                'max-AE-F count': len(max_aef_set),
                'LCMD vs max-AE-F overlap': len(lcmd_set & max_aef_set),
                'DIRECT vs max-AE-F overlap': len(direct_set & max_aef_set),
            }
        )
    return summary


def plot_lcmd_direct_umap(
    embedding: np.ndarray,
    lcmd_idx: np.ndarray,
    direct_idx: np.ndarray,
    aef: np.ndarray | None,
    run_dir: Path,
    *,
    model_label: str,
    feature_label: str,
    output_stem: str,
):
    fig, ax = plt.subplots(figsize=(8.5, 6.8), constrained_layout=True)

    color_lcmd = '#D55E00'
    color_direct = '#009E73'
    color_max_aef = '#0072B2'
    color_overlap = '#1A1A1A'

    if aef is None:
        ax.scatter(
            embedding[:, 0],
            embedding[:, 1],
            s=14,
            c='#B8BDC7',
            alpha=0.65,
            linewidths=0,
            rasterized=True,
            label='pool_set',
        )
    else:
        vmin = float(np.percentile(aef, 1.0))
        vmax = float(max(np.percentile(aef, 99.0), vmin + 1e-12))
        sc = ax.scatter(
            embedding[:, 0],
            embedding[:, 1],
            c=aef,
            cmap='Greys',
            s=14,
            alpha=0.82,
            linewidths=0,
            vmin=vmin,
            vmax=vmax,
            rasterized=True,
            label='pool_set',
        )
        cbar = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.045)
        cbar.set_label(r'Mean absolute force error, AE-F (eV/$\AA$)')

        max_aef_idx = np.argsort(aef)[-len(lcmd_idx):]
        ax.scatter(
            embedding[max_aef_idx, 0],
            embedding[max_aef_idx, 1],
            s=38,
            c=color_max_aef,
            alpha=0.36,
            linewidths=0,
            marker='D',
            zorder=4,
            label=f'max AE-F (n={len(max_aef_idx)})',
        )

    ax.scatter(
        embedding[lcmd_idx, 0],
        embedding[lcmd_idx, 1],
        s=52,
        facecolors='none',
        edgecolors=color_lcmd,
        alpha=0.72,
        linewidths=1.1,
        marker='o',
        zorder=6,
        label=f'LCMD (n={len(lcmd_idx)})',
    )
    ax.scatter(
        embedding[direct_idx, 0],
        embedding[direct_idx, 1],
        s=56,
        facecolors='none',
        edgecolors=color_direct,
        alpha=0.72,
        linewidths=1.1,
        marker='s',
        zorder=7,
        label=f'DIRECT-BIRCH (n={len(direct_idx)})',
    )

    overlap_idx = np.array(sorted(set(lcmd_idx.tolist()) & set(direct_idx.tolist())), dtype=int)
    if len(overlap_idx) > 0:
        ax.scatter(
            embedding[overlap_idx, 0],
            embedding[overlap_idx, 1],
            s=26,
            c=color_overlap,
            alpha=0.58,
            linewidths=0,
            marker='x',
            zorder=8,
            label=f'LCMD ∩ DIRECT (n={len(overlap_idx)})',
        )

    ax.set_title(f'{model_label} {feature_label}: LCMD vs DIRECT-BIRCH')
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.3)
    ax.legend(loc='best')

    plot_png = run_dir / f'{output_stem}.png'
    plot_pdf = run_dir / f'{output_stem}.pdf'
    fig.savefig(plot_png, dpi=300, bbox_inches='tight')
    fig.savefig(plot_pdf, dpi=300, bbox_inches='tight')
    return fig, plot_png, plot_pdf


In [ ]:
feature_key = 'llg-sketch'
lcmd_idx = read_selected(run_dir / 'selected_lcmd_greedy.json')
direct_idx = read_selected(run_dir / 'selected_direct_birch.json')

with h5py.File(features_h5, 'r') as handle:
    features = handle[f'features/{feature_key}/data'][0]

embedding = UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.12,
    metric='cosine',
    random_state=42,
).fit_transform(features)

aef = None
if has_reference_forces(pool_set):
    aef = compute_aef_from_adapter(model, pool_set=pool_set, batch_size=8)

matgl_summary = summarize_selection_overlap(lcmd_idx, direct_idx, aef)
print('features shape:', features.shape)
print('embedding shape:', embedding.shape)
print('has AE-F:', aef is not None)
matgl_summary


In [ ]:
fig, plot_png, plot_pdf = plot_lcmd_direct_umap(
    embedding,
    lcmd_idx,
    direct_idx,
    aef,
    run_dir,
    model_label='MatGL TensorNet',
    feature_label=feature_key,
    output_stem='umap_lcmd_direct_vs_max_aef',
)
plt.show()

print('saved:', plot_png)
print('saved:', plot_pdf)


## 7. A complete `curator-select` YAML config for a MatGL model

If you do not want to build `GeneralActiveLearning(...)` manually in Python, you can do the same job with a normal `curator-select` YAML file.

This is the complete config pattern you need for MatGL TensorNet selection.


In [ ]:
yaml_text = f"""_convert_: all
cfg: null
seed: 123
run_path: {run_dir}
model_path: matgl:TensorNet-PES-MatPES-PBE-2025.2-m
device: {device}
train_set: null
pool_set: {pool_set}
data_url: null
data_batch_size: 4
select_batch_size: 100
method: direct_birch
selection_kwargs:
  n_clusters: 100
  k: 1
  threshold: 0.5
  weighting_pcs: true
  selection_criteria: center
selection_feature: llg-sketch
feature_specs:
  - preset: llg-sketch
    num_features: 256
    seed: 123
target_layer: final_layer
save_features: true
save_selected_features: null
export_normalized_features: true
save_images: false
compute_features_only: false
target_domain: null
transforms: []
"""

print(yaml_text)


In [ ]:
yaml_path = run_dir / 'select_matgl_tensornet_2025_2_m.yaml'
yaml_path.write_text(yaml_text, encoding='utf-8')

print('saved:', yaml_path)
print()
print('Run this from a shell:')
print(f'curator-select cfg={yaml_path}')


That YAML is enough to use a MatGL model for pool selection in CURATOR.

The two most important lines are:

- `model_path: matgl:TensorNet-PES-MatPES-PBE-2025.2-m`
- `target_layer: final_layer`

The first tells CURATOR to load an external MatGL model through the adapter.
The second tells CURATOR where to attach hooks for feature extraction.


## 8. Run a full official MACE runtime selection

This section uses the same full `pool_set` as the MatGL example and loads an official MACE checkpoint directly through `mace:...?runtime=official`. It runs complete LCMD and DIRECT-BIRCH selections with `llg-sketch` features, then writes selection JSON files and `features.h5` under the notebook run directory.


In [ ]:
mace_model_url = 'https://huggingface.co/mace-foundations/mace-mp-0/resolve/main/mace-mp-0b3-medium.model'
mace_model_spec = f'mace:{mace_model_url}?runtime=official'
mace_run_dir = notebook_dir / 'runs' / 'mace_official_full_select'
mace_run_dir.mkdir(parents=True, exist_ok=True)

mace_device = device
mace_feature_specs = [
    {
        'preset': 'llg-sketch',
        'num_features': 256,
        'seed': 123,
    }
]
mace_selection_feature = 'llg-sketch'
mace_result = {
    'mace_model_url': mace_model_url,
    'mace_model_spec': mace_model_spec,
    'pool_set': str(pool_set),
}

mace_model = load_models(
    mace_model_spec,
    device=mace_device,
    load_compiled=False,
)[0]
mace_features_h5 = mace_run_dir / 'features_llg.h5'
mace_al = GeneralActiveLearning(
    models=[mace_model],
    selection='lcmd_greedy',
    feature_specs=mace_feature_specs,
    selection_feature=mace_selection_feature,
    target_layer='readout',
    batch_size=4,
    device=mace_device,
    dataset_cutoff=mace_model.representation.cutoff,
    transforms=[],
    save_features=str(mace_features_h5),
    target_domain=None,
)
mace_selected = mace_al.select(
    pool_set=str(pool_set),
    train_set=None,
    select_batch_size=100,
    save_json=str(mace_run_dir / 'selected_lcmd_greedy.json'),
    save_images=None,
    save_selected_features=None,
    normalize_features=True,
    compute_features_only=False,
)

mace_direct_al = GeneralActiveLearning(
    models=[mace_model],
    selection='direct_birch',
    selection_kwargs=direct_selection_kwargs,
    feature_specs=mace_feature_specs,
    selection_feature=mace_selection_feature,
    target_layer='readout',
    batch_size=4,
    device=mace_device,
    dataset_cutoff=mace_model.representation.cutoff,
    transforms=[],
    save_features=str(mace_features_h5),
    target_domain=None,
)
mace_direct_selected = mace_direct_al.select(
    pool_set=str(pool_set),
    train_set=None,
    select_batch_size=100,
    save_json=str(mace_run_dir / 'selected_direct_birch.json'),
    save_images=None,
    save_selected_features=None,
    normalize_features=True,
    compute_features_only=False,
)

mace_result.update(
    {
        'loaded_object': mace_model.__class__.__name__,
        'core_model': mace_model.model.__class__.__name__,
        'cutoff': mace_model.representation.cutoff,
        'target_layer': mace_al.target_layer,
        'selection_feature': mace_al.selection_feature,
        'direct_selection_kwargs': direct_selection_kwargs,
        'select_batch_size': 100,
        'lcmd_selected_count': len(mace_selected),
        'direct_selected_count': len(mace_direct_selected),
        'lcmd_first_ten_selected': mace_selected[:10],
        'direct_first_ten_selected': mace_direct_selected[:10],
        'features_h5': str(mace_features_h5),
        'lcmd_selected_json': str(mace_run_dir / 'selected_lcmd_greedy.json'),
        'direct_selected_json': str(mace_run_dir / 'selected_direct_birch.json'),
    }
)

mace_result


## 9. Visualize the full MACE selection

This plot mirrors the MatGL visualization above: it embeds the MACE `llg-sketch` features, colors the pool by MACE AE-F, and compares the MACE `lcmd_greedy` and `direct_birch` selections with the structures that have the largest MACE AE-F.


In [ ]:
mace_feature_key = 'llg-sketch'
mace_lcmd_idx = read_selected(mace_run_dir / 'selected_lcmd_greedy.json')
mace_direct_idx = read_selected(mace_run_dir / 'selected_direct_birch.json')

with h5py.File(mace_features_h5, 'r') as handle:
    mace_features = handle[f'features/{mace_feature_key}/data'][0]

mace_embedding = UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.12,
    metric='cosine',
    random_state=42,
).fit_transform(mace_features)

mace_aef = None
if has_reference_forces(pool_set):
    mace_aef = compute_aef_from_curator_model(mace_model, pool_set=pool_set, batch_size=4)

fig, mace_plot_png, mace_plot_pdf = plot_lcmd_direct_umap(
    mace_embedding,
    mace_lcmd_idx,
    mace_direct_idx,
    mace_aef,
    mace_run_dir,
    model_label='MACE official',
    feature_label=mace_feature_key,
    output_stem='umap_lcmd_direct_vs_max_aef',
)
plt.show()

print('mace features shape:', mace_features.shape)
print('mace embedding shape:', mace_embedding.shape)
print('has AE-F:', mace_aef is not None)
print('saved:', mace_plot_png)
print('saved:', mace_plot_pdf)
summarize_selection_overlap(mace_lcmd_idx, mace_direct_idx, mace_aef)
